# YOLOv8 Cup Orientation Training - Google Colab
# Created by Mahmoud Ayoub
# MDU November 2025

Train your cup orientation model with free Tesla T4 GPU!

**Time: 1-2 hours**  
**Expected mAP: 99.5%**

---

## Setup Instructions:

1. **Enable GPU**: Runtime → Change runtime type → T4 GPU
2. **Run all cells**: Runtime → Run all
3. **Download model**: After training, download `best.pt` from Files panel

---

## Step 1: Check GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: GPU not enabled!")
    print("Go to: Runtime → Change runtime type → Hardware accelerator → T4 GPU")

## Step 2: Install Dependencies

In [ ]:
!pip install ultralytics roboflow -q
print("✓ Ultralytics and Roboflow installed!")

## Step 3: Download Dataset from Roboflow

**Option A: Use Roboflow API (Easiest)**

In [ ]:
from roboflow import Roboflow

# Your Roboflow credentials
API_KEY = "PPE7YGMD9vIcPgUUXiI8"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("cup-orientation-detection").project("final-version-jdfkm")
dataset = project.version(1).download("yolov8")

print(f"\n[OK] Dataset downloaded to: {dataset.location}")
print(f"[OK] data.yaml path: {dataset.location}/data.yaml")

**Option B: Upload Dataset Manually (if API doesn't work)**

1. Upload your dataset ZIP to Google Drive
2. Mount Drive and extract:

```python
from google.colab import drive
drive.mount('/content/drive')

!unzip '/content/drive/MyDrive/your-dataset.zip' -d /content/dataset
dataset_location = '/content/dataset/New Orientation.v3i.yolov8'
```

## Step 4: Train Model

In [ ]:
from ultralytics import YOLO

# Load pretrained model (Medium size to match Roboflow)
model = YOLO('yolov8m.pt')

print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70)
print("Model: YOLOv8m (Medium)")
print("Expected time: 1-2 hours")
print("Target mAP: 99.5%")
print("="*70 + "\n")

# Train
results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,  # Full batch size with Tesla T4 GPU
    device=0,  # Use GPU
    
    # Early stopping
    patience=50,
    
    # Save settings
    save=True,
    save_period=10,
    
    # Disable augmentation (already done in Roboflow)
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    flipud=0.0,
    fliplr=0.0,
    
    # Logging
    plots=True,
    name='cup_orientation'
)

print("\n" + "="*70)
print("🎉 TRAINING COMPLETE! 🎉")
print("="*70)

## Step 5: Validate Model

In [ ]:
# Load best model
best_model = YOLO('runs/detect/cup_orientation/weights/best.pt')

# Validate
metrics = best_model.val()

print("\n" + "="*70)
print("FINAL METRICS")
print("="*70)
print(f"mAP@50: {metrics.box.map50:.4f} ({metrics.box.map50*100:.2f}%)")
print(f"mAP@50-95: {metrics.box.map:.4f} ({metrics.box.map*100:.2f}%)")
print(f"Precision: {metrics.box.mp:.4f} ({metrics.box.mp*100:.2f}%)")
print(f"Recall: {metrics.box.mr:.4f} ({metrics.box.mr*100:.2f}%)")
print("="*70)

## Step 6: Download Trained Model

In [ ]:
from google.colab import files
import shutil
import os

# Copy best model
shutil.copy(
    "runs/detect/cup_orientation/weights/best.pt",
    "best_cup_orientation.pt"
)

print("[OK] Preparing downloads...")

# Download 1: best.pt model
print("\n[1/2] Downloading best.pt model...")
files.download("best_cup_orientation.pt")
print("[OK] Model downloaded!")

# Download 2: Zip entire training folder
print("\n[2/2] Creating training results zip...")
shutil.make_archive(
    "cup_orientation_training_results",
    "zip",
    "runs/detect/cup_orientation"
)
print("[OK] Zip created!")

print("\nDownloading training results folder...")
files.download("cup_orientation_training_results.zip")

print("\n" + "="*70)
print("DOWNLOADS COMPLETE!")
print("="*70)
print("\nYou have downloaded:")
print("1. best_cup_orientation.pt - Trained model")
print("2. cup_orientation_training_results.zip - Full results folder")
print("\nNext steps:")
print("1. Rename best_cup_orientation.pt to best.pt")
print("2. Copy best.pt to your project folder")
print("3. Extract zip to view training plots/metrics")
print("="*70)


## Optional: View Training Results

In [ ]:
from IPython.display import Image, display
import os

results_dir = 'runs/detect/cup_orientation'

print("Training Results:\n")

# Show key plots
plots = [
    'results.png',
    'confusion_matrix.png',
    'F1_curve.png',
    'PR_curve.png'
]

for plot in plots:
    plot_path = os.path.join(results_dir, plot)
    if os.path.exists(plot_path):
        print(f"\n{plot}:")
        display(Image(filename=plot_path, width=800))
    else:
        print(f"⚠️ {plot} not found")

## Optional: Test on Sample Image

In [ ]:
# Test inference on a validation image
import glob

# Get a test image
test_images = glob.glob(f'{dataset.location}/test/images/*.jpg')
if test_images:
    test_image = test_images[0]
    
    # Run inference
    results = best_model.predict(test_image, conf=0.5)
    
    # Display
    print("\nTest Image Detection:")
    results[0].show()
else:
    print("No test images found")